# Phase 6.4B — Field-specific SAP Evidence Retrieval
Zero new LLM API calls. Reuse verified Phase 5.2 SAP body chunks and Phase 6.4A extraction inputs. Compare matching baseline, field retrieval with matched chunk count, and expanded field retrieval. Local BGE encoding only; model weights may be downloaded on first use. No template candidate, historical claim, or gold label is supplied to retrieval. No specification extraction or generation.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, shutil, json
REPO_URL='https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH='simplified-rag-v3'
ROOT=Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC=ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_phase6_field_retrieval.py')],check=True)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from tfl_agent.agent.phase6_field_retrieval import (
    PREFLIGHT_NAME,PREFLIGHT_SHA,HISTORICAL_NAME,HISTORICAL_SHA,MODEL,
    sha_file,load_preflight,query_plan,run_field_retrieval)
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
def frozen_file(folder,name,sha):
    p=BASE/folder/name
    if p.is_file() and sha_file(p)==sha:
        return p
    matches=[x for x in p.parent.glob('*.zip') if sha_file(x)==sha]
    if len(matches)!=1:
        raise FileNotFoundError(f'Expected {p} with SHA256 {sha}; matching archives: {matches}')
    return matches[0]
PREFLIGHT=frozen_file('Phase6331_Simplified_Replay_Outputs',PREFLIGHT_NAME,PREFLIGHT_SHA)
HISTORICAL=frozen_file('Phase52_Outputs',HISTORICAL_NAME,HISTORICAL_SHA)
inputs=load_preflight(PREFLIGHT)
print('Verified preflight:',PREFLIGHT)
print('Verified historical body archive:',HISTORICAL)
print('Cases:',len(inputs),'| queries per case:',len(query_plan(inputs[0]['payload']['target'])))
print('New LLM calls authorized: 0')

In [ ]:
from tfl_agent.tools.sap_hybrid import LocalEncoder
import torch
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
encoder=LocalEncoder(model_name=MODEL,device=DEVICE)
print('Local encoder:',MODEL,'| device:',DEVICE)
CACHE=Path('/content/Phase64B_Cache')/RUN_ID
OUT=Path('/content/Phase64B_Field_Retrieval')/RUN_ID
report,packages=run_field_retrieval(PREFLIGHT,HISTORICAL,CACHE,OUT,encoder)
assert report['new_llm_calls']==0 and report['cases']==6 and len(packages)==18
assert all(set(r['payload'])=={'target','sap_evidence','allowed_fields'} for r in packages)
assert not report['generation_authorized']
print(json.dumps(report,indent=2))

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.DataFrame(report['metrics']).T)
budget=pd.read_csv(OUT/'Evidence_Budget.csv')
display(budget[['study_id','entry_id','arm','chunks','chars','new_vs_matching']])
coverage=pd.read_csv(OUT/'Field_Group_Coverage.csv')
display(coverage.pivot_table(index=['study_id','entry_id','group'],columns='arm',values='group_candidate_hits',aggfunc='first'))
print('Group candidate hits do NOT establish semantic field coverage.')
print('Historical quote overlap is NOT independent recall.')
DEST=BASE/'Phase64B_Field_Retrieval_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print('Files:',sorted(p.name for p in OUT.iterdir()))
print('Share only if authorized to disclose SAP excerpts.')

## Interpretation
- All 12 fields map to five query groups. Each has two queries; general study context is searched separately where useful. Search covers the full verified SAP body without hard topic filtering.
- Matching baseline reuses original evidence; field_count_matched uses the same number of chunks if enough candidates exist, not equal token length; field_expanded is limited to 20 unique chunk IDs and 24,000 evidence characters per case. Small count budgets can favor earlier groups; consult allocation coverage. Adjacent overlapping spans can still repeat text.
- Field_Query_Results.json preserves query rankings and scopes. Extraction_Evidence_Packages.json contains three independent extraction inputs per target with verified source spans, no library context or historical claims.
- Pooled_Evidence_Review.csv is unlabeled, not a semantic classifier. Historical_Claim_Overlap.csv compares unique previously accepted/review target claims, not independent gold. No new specification claims or DOCX are generated.